In [ ]:
# ============================================================
# SELF-TRAINING ON MNIST
# 90/10 TO 10/90 SPLITS
# PROGRESSIVE THRESHOLDING
# FINAL VERSION - TYPE + CPU/GPU ERRORS FIXED
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    DataLoader,
    Dataset,
    TensorDataset,
    ConcatDataset
)

from torchvision import datasets, transforms

import numpy as np
import pandas as pd
import random
import gc


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

BATCH_SIZE = 128

# 3 epochs keeps the experiment reasonably fast.
EPOCHS_INITIAL = 3
EPOCHS_RETRAIN = 3


# ============================================================
# 2. THRESHOLD SCHEDULE
# ============================================================

threshold_schedule = {

    0.90: [0.98],

    0.80: [0.98],

    0.70: [0.98],

    0.60: [0.98, 0.95],

    0.50: [0.98, 0.95],

    0.40: [0.98, 0.95],

    0.30: [0.98, 0.95, 0.90],

    0.20: [0.98, 0.95, 0.90],

    0.10: [0.98, 0.95, 0.90]
}


# ============================================================
# 3. MNIST DATASET
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print(
    "Total training samples:",
    len(train_dataset)
)

print(
    "Total test samples:",
    len(test_dataset)
)


# ============================================================
# 4. IMPORTANT:
# LABELED DATASET THAT ALWAYS RETURNS TENSOR LABELS
# ============================================================

class LabeledSubset(Dataset):

    def __init__(
        self,
        dataset,
        indices
    ):

        self.dataset = dataset
        self.indices = list(indices)

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, position):

        index = self.indices[position]

        image, label = self.dataset[index]

        # Make absolutely sure label is a Tensor
        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return image, label


# ============================================================
# 5. CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self):

        super(CNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 7 * 7,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                10
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ============================================================
# 6. TRAINING FUNCTION
# ============================================================

def train_model(
    model,
    dataset,
    epochs
):

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
                *
                images.size(0)
            )

        avg_loss = (
            running_loss
            /
            len(dataset)
        )

        print(
            f"      Epoch "
            f"{epoch + 1}/{epochs} "
            f"Loss: {avg_loss:.4f}"
        )

    return model


# ============================================================
# 7. TEST ACCURACY
# ============================================================

def test_accuracy(model):

    loader = DataLoader(

        test_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )

    model.eval()

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        100.0
        *
        correct
        /
        total
    )


# ============================================================
# 8. DATASET THAT RETURNS IMAGE + ORIGINAL INDEX
# ============================================================

class IndexedSubset(Dataset):

    def __init__(
        self,
        dataset,
        indices
    ):

        self.dataset = dataset

        self.indices = list(indices)

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, position):

        original_index = (
            self.indices[position]
        )

        image, _ = self.dataset[
            original_index
        ]

        return (
            image,
            original_index
        )


# ============================================================
# 9. GENERATE PSEUDO-LABELS
# ============================================================

def generate_pseudo_labels(
    model,
    dataset,
    unlabeled_indices,
    threshold
):

    if len(unlabeled_indices) == 0:

        return (
            [],
            torch.empty(
                (0, 1, 28, 28)
            ),
            torch.empty(
                (0,),
                dtype=torch.long
            ),
            0
        )

    indexed_dataset = IndexedSubset(
        dataset,
        unlabeled_indices
    )

    loader = DataLoader(

        indexed_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )

    model.eval()

    selected_indices = []

    selected_images = []

    selected_labels = []

    with torch.no_grad():

        for images, original_indices in loader:

            # --------------------------------------------
            # Move images to GPU
            # --------------------------------------------

            images_gpu = images.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images_gpu
            )

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            confidence, predictions = torch.max(
                probabilities,
                dim=1
            )

            mask = (
                confidence >= threshold
            )

            # --------------------------------------------
            # FIX CPU/GPU INDEXING
            # --------------------------------------------

            mask_cpu = mask.cpu()

            if mask_cpu.any():

                # original_indices = CPU
                selected_indices.extend(
                    original_indices[
                        mask_cpu
                    ].tolist()
                )

                # images = CPU
                selected_images.append(
                    images[
                        mask_cpu
                    ]
                )

                # predictions = GPU
                selected_labels.append(
                    predictions[
                        mask
                    ].cpu()
                )

    # ========================================================
    # COMBINE SELECTED SAMPLES
    # ========================================================

    if len(selected_images) > 0:

        selected_images = torch.cat(
            selected_images,
            dim=0
        )

        selected_labels = torch.cat(
            selected_labels,
            dim=0
        )

    else:

        selected_images = torch.empty(
            (0, 1, 28, 28)
        )

        selected_labels = torch.empty(
            (0,),
            dtype=torch.long
        )

    rejected = (
        len(unlabeled_indices)
        -
        len(selected_indices)
    )

    return (
        selected_indices,
        selected_images,
        selected_labels,
        rejected
    )


# ============================================================
# 10. RUN ALL 9 SPLITS
# ============================================================

all_results = []

total_samples = len(
    train_dataset
)


for labeled_fraction, thresholds in threshold_schedule.items():

    # ========================================================
    # CORRECT SPLIT NAME
    # ========================================================

    labeled_percent = int(
        labeled_fraction * 100
    )

    unlabeled_percent = (
        100 - labeled_percent
    )

    split_name = (
        f"{labeled_percent}/"
        f"{unlabeled_percent}"
    )

    print("\n" + "=" * 70)

    print(
        f"EXPERIMENT: "
        f"{split_name} "
        f"(LABELED / UNLABELED)"
    )

    print(
        f"Threshold schedule: "
        f"{thresholds}"
    )

    print("=" * 70)


    # ========================================================
    # CREATE RANDOM SPLIT
    # ========================================================

    indices = np.arange(
        total_samples
    )

    rng = np.random.default_rng(
        SEED
    )

    rng.shuffle(indices)

    num_labeled = int(
        total_samples
        *
        labeled_fraction
    )

    labeled_indices = (
        indices[
            :num_labeled
        ]
        .tolist()
    )

    unlabeled_indices = (
        indices[
            num_labeled:
        ]
        .tolist()
    )

    print(
        f"Initial labeled samples: "
        f"{len(labeled_indices)}"
    )

    print(
        f"Initial unlabeled samples: "
        f"{len(unlabeled_indices)}"
    )


    # ========================================================
    # LABELED DATASET
    # ========================================================

    labeled_dataset = LabeledSubset(
        train_dataset,
        labeled_indices
    )


    # ========================================================
    # CREATE MODEL
    # ========================================================

    model = CNN().to(device)


    # ========================================================
    # INITIAL TRAINING
    # ========================================================

    print("\nInitial training...")

    model = train_model(
        model,
        labeled_dataset,
        EPOCHS_INITIAL
    )

    initial_accuracy = test_accuracy(
        model
    )

    print(
        f"Initial Test Accuracy: "
        f"{initial_accuracy:.2f}%"
    )


    # ========================================================
    # SAVE INITIAL RESULT
    # ========================================================

    all_results.append({

        "split":
            split_name,

        "threshold":
            "Initial",

        "pseudo_labels_added":
            0,

        "total_pseudo_labels":
            0,

        "remaining_unlabeled":
            len(unlabeled_indices),

        "labeled_samples":
            len(labeled_indices),

        "accuracy_percent":
            round(
                initial_accuracy,
                2
            )
    })


    # ========================================================
    # STORAGE FOR PSEUDO-LABELED DATA
    # ========================================================

    accumulated_pseudo_images = []

    accumulated_pseudo_labels = []

    total_pseudo_labels = 0


    # ========================================================
    # SELF-TRAINING STAGES
    # ========================================================

    for stage_number, threshold in enumerate(
        thresholds,
        start=1
    ):

        print(
            f"\n--- Self-Training Stage "
            f"{stage_number} ---"
        )

        print(
            f"Threshold = {threshold}"
        )


        # ====================================================
        # GENERATE PSEUDO-LABELS
        # ====================================================

        (
            selected_indices,
            pseudo_images,
            pseudo_labels,
            rejected
        ) = generate_pseudo_labels(

            model,

            train_dataset,

            unlabeled_indices,

            threshold
        )

        num_selected = len(
            selected_indices
        )

        print(
            f"Pseudo-labels selected: "
            f"{num_selected}"
        )

        print(
            f"Rejected / remaining: "
            f"{rejected}"
        )


        # ====================================================
        # STOP IF NOTHING SELECTED
        # ====================================================

        if num_selected == 0:

            print(
                "No confident samples found."
            )

            print(
                "Stopping this split."
            )

            break


        # ====================================================
        # STORE PSEUDO-LABELED DATA
        # ====================================================

        accumulated_pseudo_images.append(
            pseudo_images
        )

        accumulated_pseudo_labels.append(
            pseudo_labels
        )

        total_pseudo_labels += (
            num_selected
        )


        # ====================================================
        # REMOVE SELECTED SAMPLES
        # ====================================================

        selected_set = set(
            selected_indices
        )

        unlabeled_indices = [

            idx

            for idx in unlabeled_indices

            if idx not in selected_set

        ]


        # ====================================================
        # COMBINE ALL PSEUDO-LABELED DATA
        # ====================================================

        pseudo_images_all = torch.cat(
            accumulated_pseudo_images,
            dim=0
        )

        pseudo_labels_all = torch.cat(
            accumulated_pseudo_labels,
            dim=0
        )

        # Make sure labels are definitely LONG tensors
        pseudo_labels_all = (
            pseudo_labels_all
            .long()
        )

        pseudo_dataset = TensorDataset(

            pseudo_images_all,

            pseudo_labels_all

        )


        # ====================================================
        # COMBINE REAL + PSEUDO LABELLED DATA
        # ====================================================

        combined_dataset = ConcatDataset([

            labeled_dataset,

            pseudo_dataset

        ])


        # ====================================================
        # RETRAIN
        # ====================================================

        print(
            "Retraining with labeled + "
            "pseudo-labeled data..."
        )

        model = train_model(

            model,

            combined_dataset,

            EPOCHS_RETRAIN

        )


        # ====================================================
        # TEST
        # ====================================================

        accuracy = test_accuracy(
            model
        )

        print(
            f"Test Accuracy after "
            f"threshold {threshold}: "
            f"{accuracy:.2f}%"
        )

        print(
            f"Total pseudo-labels: "
            f"{total_pseudo_labels}"
        )

        print(
            f"Remaining unlabeled: "
            f"{len(unlabeled_indices)}"
        )


        # ====================================================
        # SAVE RESULT
        # ====================================================

        all_results.append({

            "split":
                split_name,

            "threshold":
                threshold,

            "pseudo_labels_added":
                num_selected,

            "total_pseudo_labels":
                total_pseudo_labels,

            "remaining_unlabeled":
                len(unlabeled_indices),

            "labeled_samples":
                (
                    len(labeled_indices)
                    +
                    total_pseudo_labels
                ),

            "accuracy_percent":
                round(
                    accuracy,
                    2
                )
        })


    # ========================================================
    # CLEAR GPU MEMORY
    # ========================================================

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 11. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)

print("\n" + "=" * 70)

print(
    "FINAL SELF-TRAINING RESULTS"
)

print("=" * 70)

display(
    results_df
)


# ============================================================
# 12. VERIFY SPLIT LABELS
# ============================================================

print(
    "\nCorrect split labels:"
)

print(
    results_df[
        "split"
    ].unique()
)


# ============================================================
# 13. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "self_training_results_corrected.csv",
    index=False
)

print(
    "\nResults saved as:"
)

print(
    "self_training_results_corrected.csv"
)

Device: cuda
Total training samples: 60000
Total test samples: 10000

EXPERIMENT: 90/10 (LABELED / UNLABELED)
Threshold schedule: [0.98]
Initial labeled samples: 54000
Initial unlabeled samples: 6000

Initial training...
      Epoch 1/3 Loss: 0.2652
      Epoch 2/3 Loss: 0.0640
      Epoch 3/3 Loss: 0.0443
Initial Test Accuracy: 98.70%

--- Self-Training Stage 1 ---
Threshold = 0.98
Pseudo-labels selected: 5442
Rejected / remaining: 558
Retraining with labeled + pseudo-labeled data...
      Epoch 1/3 Loss: 0.0355
      Epoch 2/3 Loss: 0.0239
      Epoch 3/3 Loss: 0.0192
Test Accuracy after threshold 0.98: 99.07%
Total pseudo-labels: 5442
Remaining unlabeled: 558

EXPERIMENT: 80/20 (LABELED / UNLABELED)
Threshold schedule: [0.98]
Initial labeled samples: 48000
Initial unlabeled samples: 12000

Initial training...
      Epoch 1/3 Loss: 0.2699
      Epoch 2/3 Loss: 0.0641
      Epoch 3/3 Loss: 0.0439
Initial Test Accuracy: 98.71%

--- Self-Training Stage 1 ---
Threshold = 0.98
Pseudo-labe

,split,threshold,pseudo_labels_added,total_pseudo_labels,remaining_unlabeled,labeled_samples,accuracy_percent
0,90/10,Initial,0,0,6000,54000,98.70
1,90/10,0.98,5442,5442,558,59442,99.07
2,80/20,Initial,0,0,12000,48000,98.71
3,80/20,0.98,10931,10931,1069,58931,98.81
4,70/30,Initial,0,0,18000,42000,98.71
5,70/30,0.98,16250,16250,1750,58250,98.72
6,60/40,Initial,0,0,24000,36000,98.03
7,60/40,0.98,20981,20981,3019,56981,98.55
8,60/40,0.95,2106,23087,913,59087,98.67
9,50/50,Initial,0,0,30000,30000,98.37



Correct split labels:
['90/10' '80/20' '70/30' '60/40' '50/50' '40/60' '30/70' '20/80' '10/90']

Results saved as:
self_training_results_corrected.csv
